In [3]:
# Load the libraries
import re
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

In [11]:
#1. Load the dataset
df= pd.read_csv('Desktop/SS/used_cars.csv')

In [14]:
#Datashape
print(f"Dataset shape:{df.shape}\n")

Dataset shape:(4009, 12)



In [19]:
#Head of the data
df.head(100)

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"
...,...,...,...,...,...,...,...,...,...,...,...,...
95,BMW,435 Gran Coupe i,2016,"76,009 mi.",Gasoline,300.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,8-Speed A/T,White,Beige,None reported,Yes,"$24,950"
96,Jeep,Grand Cherokee Summit,2020,"31,349 mi.",Gasoline,5.7L V8 16V MPFI OHV,8-Speed Automatic,Granite Crystal Clearcoat Metallic,Black,None reported,NaN,"$39,366"
97,Lexus,RC 350 Base,2015,"48,000 mi.",Gasoline,306.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,Transmission w/Dual Shift Mode,Orange,Red,None reported,Yes,"$32,900"
98,BMW,525 i,2005,"91,420 mi.",Gasoline,215.0HP 2.5L Straight 6 Cylinder Engine Gasoli...,6-Speed A/T,Black,Black,None reported,Yes,"$8,900"


In [20]:

df.describe()

,model_year
count,4009.000000
mean,2015.515590
std,6.104816
min,1974.000000
25%,2012.000000
50%,2017.000000
75%,2020.000000
max,2024.000000


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   brand         4009 non-null   str  
 1   model         4009 non-null   str  
 2   model_year    4009 non-null   int64
 3   milage        4009 non-null   str  
 4   fuel_type     3839 non-null   str  
 5   engine        4009 non-null   str  
 6   transmission  4009 non-null   str  
 7   ext_col       4009 non-null   str  
 8   int_col       4009 non-null   str  
 9   accident      3896 non-null   str  
 10  clean_title   3413 non-null   str  
 11  price         4009 non-null   str  
dtypes: int64(1), str(11)
memory usage: 376.0 KB


In [31]:
#Clean the data
df['price']= df['price'].astype(str)

In [32]:
# Step 2: Clean the data
# Remove the dollar sign and commas, then convert to a float
df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

In [33]:
#Step 3 squashes extreme luxury car outliers so they dont break your RSME score
y=np.log1p(df['price'])

In [35]:
# Clean and convert mileage to float
df['milage'] = df['milage'].str.replace('mi.', '', regex=False).str.replace(',', '', regex=False).astype(float)

In [38]:
#Calculate car Age 
df['Car_age']=2026 - df['model_year']

In [41]:
# Calculate Mileage intensity
df['mileage_per_year'] = df['milage'] / (df['car_age'] + 1)

In [42]:
# Advanced Engine extraction
def extract_engine_fatures(engine_str):engine_str=str(engine_str).lower()

In [45]:
# Extract Horsepower
import re
import numpy as np
# Fixed version
engine_str = "The car has 150hp engine"
hp_match = re.search(r'(\d+\.?\d*)hp', engine_str)
hp = float(hp_match.group(1)) if hp_match else np.nan

In [49]:
# Extract Engine litres
import re
import numpy as np
litres_match = re.search(r'(\d+\.?\d*)\s*(?:liter|litre|l)', engine_str, re.IGNORECASE)
liters = float(litres_match.group(1)) if litres_match else np.nan

In [48]:
import re
import numpy as np
cyl_match = re.search(r'(\d+)\s*cylinder|v(\d+)|i(\d+)', engine_str, re.IGNORECASE)
cyl = np.nan
if cyl_match:
    for group in cyl_match.groups():
        if group:
            cyl = float(group)
            break

In [62]:
# Make sure everything is lowercase to avoid missing "Turbo" vs "turbo"
print(df.columns.tolist())
#Keep the original
engine_series = df['engine'].str.lower()



['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price', 'car_age', 'Car_age', 'mileage_per_year']


In [63]:
#Etract turbo from supercharged
df['is_turbo'] = engine_series.str.contains('turbo').fillna(False).astype(int)
df['is_supercharged'] = engine_series.str.contains('supercharged').fillna(False).astype(int)

In [65]:

print(df[['engine', 'is_turbo', 'is_supercharged']].head())

                                              engine  is_turbo  \
0  300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...         0   
1                               3.8L V6 24V GDI DOHC         0   
2                                     3.5 Liter DOHC         0   
3  354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...         0   
4                         2.0L I4 16V GDI DOHC Turbo         1   

   is_supercharged  
0                0  
1                0  
2                0  
3                0  
4                0  


In [70]:
print(df.columns.tolist())

['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price', 'car_age', 'Car_age', 'mileage_per_year', 'is_turbo', 'is_supercharged']


In [80]:
# 1. Fill the base columns with their respective medians
base_cols = ['milage', 'Car_age']
df[base_cols] = df[base_cols].fillna(df[base_cols].median())



In [87]:
# Missing  Values
text_cols = ['fuel_type', 'transmission', 'ext_col', 'int_col']
# Find the missing values and fill them with the mode
for col in text_cols:
# The safest, explicit way using .fillna()
    df[col] = df[col].fillna(df[col].mode().dropna().iloc[0])


In [88]:
# check
print(df.isnull().sum())

brand                 0
model                 0
model_year            0
milage                0
fuel_type             0
engine                0
transmission          0
ext_col               0
int_col               0
accident            113
clean_title         596
price                 0
car_age               0
Car_age               0
mileage_per_year      0
is_turbo              0
is_supercharged       0
dtype: int64


In [91]:
final_cols = ['accident', 'clean_title']
for col in final_cols:
    df[col] = df[col].fillna(df[col].mode().dropna().iloc[0])

In [92]:
final_cols = ['accident', 'clean_title']
for col in final_cols:
    # returns None (or a default) instead of crashing if the mode is empty
    mode_series = df[col].mode().dropna()
    if not mode_series.empty:
        df[col] = df[col].fillna(mode_series.iloc[0])

In [93]:
#Check
print(df.isnull().sum())

brand               0
model               0
model_year          0
milage              0
fuel_type           0
engine              0
transmission        0
ext_col             0
int_col             0
accident            0
clean_title         0
price               0
car_age             0
Car_age             0
mileage_per_year    0
is_turbo            0
is_supercharged     0
dtype: int64


In [94]:
# Advanced transmission features
def extract_trans_features(trans_str): trans_str = str(trans_str).lower()

In [97]:
# Etra gear speed
import numpy as np
import pandas as pd

# 1. Lowercase once to make matching case-insensitive and robust
trans_lower = df['transmission'].str.lower()

# 2. Extract gear speeds (vectorized regex extraction)
df['trans_speeds'] = trans_lower.str.extract(r'(\d+)\s*-?\s*speed').astype(float)

# 3. Classify transmission type using vector string contains
# Note: Added word boundaries '\b' to avoid partial matches
auto_keywords = r'automatic|a/t|auto|cvt|dual shift'
manual_keywords = r'manual|m/t'

df['is_automatic'] = trans_lower.str.contains(auto_keywords, regex=True).astype(int)
df['is_manual'] = trans_lower.str.contains(manual_keywords, regex=True).astype(int)

# 4. Fill missing speeds with the median
df['trans_speeds'] = df['trans_speeds'].fillna(df['trans_speeds'].median())


In [99]:
# F. Group Luxury Brand Tiers
# Super-luxury brands command completely different pricing rules
ultra_luxury_brands = ['Bugatti', 'Rolls-Royce', 'Lamborghini', 'Ferrari', 'McLaren', 'Maserati', 'Bentley', 'Aston']
df['is_ultra_luxury'] = df['brand'].apply(lambda x: 1 if x in ultra_luxury_brands else 0)


In [102]:
# Clean corrupt/missing string markers
df['fuel_type'] = df['fuel_type'].replace(['–', 'not supported'], 'Unknown').fillna('Unknown')
df['accident'] = df['accident'].fillna('Unknown')
df['clean_title'] = df['clean_title'].fillna('Unknown')


In [104]:
# Convert remaining string text categories into numeric features using Label Encoding
categorical_cols = ['brand', 'model', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))


In [105]:
# Separate our predictive features from the target variable
X = df.drop(columns=['price'])


In [106]:
# We setup a 5-Fold Cross Validation test rig to ensure our model doesn't overfit
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = []

print("\n--- Step 5: Training Model Across 5-Fold Cross Validation ---")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
   


--- Step 5: Training Model Across 5-Fold Cross Validation ---


In [110]:
# Use HistGradientBoostingRegressor (Fast, highly accurate ensemble trees)
model = HistGradientBoostingRegressor(
    max_iter=400, 
    learning_rate=0.04, 
    max_depth=6,
    min_samples_leaf=15,
    random_state=42
)
model.fit(X_train, y_train)

# Predict and reverse log transformation (expm1 reverts log1p)
log_preds = model.predict(X_val)
preds = np.expm1(log_preds)
actuals = np.expm1(y_val)

# Ensure no tiny floating-point anomalies drop below 0
preds = np.maximum(preds, 0)

In [112]:
# Calculate RMSE for this split folder
fold_rmse = np.sqrt(mean_squared_error(actuals, preds))
rmse_scores.append(fold_rmse)

# FIXED: Ensured 'fold' matches your actual loop variable name
print(f" -> Fold {fold + 1} Validation RMSE: ${fold_rmse:,.2f}")

print("\n=========================================")
print(f"FINAL MEAN VALIDATION RMSE: ${np.mean(rmse_scores):,.2f}")
print("=========================================")

 -> Fold 5 Validation RMSE: $20,012.22

FINAL MEAN VALIDATION RMSE: $20,012.22


In [117]:

print(df.shape)


(4009, 21)


In [118]:

print(df.head())


   brand  model  model_year   milage  fuel_type  engine  transmission  \
0     14   1743        2013  51000.0          1     581            16   
1     19   1182        2021  34742.0          2     566            32   
2     27   1325        2022  22372.0          2     541            40   
3     20   1242        2015  88900.0          3     724            23   
4      3   1225        2021   9835.0          2     200            32   

   ext_col  int_col  accident  ...    price  car_age  Car_age  \
0       29       14         0  ...  10300.0       13       13   
1      185       71         0  ...  38005.0        5        5   
2       38       14         1  ...  54598.0        4        4   
3       29       14         1  ...  15500.0       11       11   
4      120       14         1  ...  34999.0        5        5   

   mileage_per_year  is_turbo  is_supercharged  trans_speeds  is_automatic  \
0       3642.857143         0                0           6.0             1   
1       5790.3

In [119]:

print(df.isnull().sum())

brand               0
model               0
model_year          0
milage              0
fuel_type           0
engine              0
transmission        0
ext_col             0
int_col             0
accident            0
clean_title         0
price               0
car_age             0
Car_age             0
mileage_per_year    0
is_turbo            0
is_supercharged     0
trans_speeds        0
is_automatic        0
is_manual           0
is_ultra_luxury     0
dtype: int64


In [120]:
from sklearn.model_selection import train_test_split

# Separate your features (X) and target variable (y)
X = df.drop(columns=['price'])  # Replace 'price' with your actual target column name
y = df['price']

# Split into 80% training and 20% validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split successfully! Training shape:", X_train.shape)



Data split successfully! Training shape: (3207, 20)


In [124]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# 1. Initialize the model
# random_state ensures you get the same results every time you run it
model = RandomForestRegressor(random_state=42, n_estimators=100)


In [125]:
# 2. Train (Fit) the model on your training data
print("Training the model... Please wait.")
model.fit(X_train, y_train)
print("Training complete!")



Training the model... Please wait.
Training complete!
